# Ruta B: Ciencia de Datos
## ICD Financiero S.A. | Optimización de la Gestión de Quejas

**Objetivo:** Identificar combinaciones de entidad, producto y motivo que concentran mayor riesgo 
regulatorio, y construir un modelo que anticipe ese riesgo.

**Dataset:** 1.044.110 registros trimestrales | 2014–2026 | Superintendencia Financiera de Colombia

In [65]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns
from pathlib import Path

# Configuración global de visualización
plt.rcParams['figure.figsize'] = (12, 5)
plt.rcParams['axes.spines.top'] = False
plt.rcParams['axes.spines.right'] = False
sns.set_palette('muted')

pd.set_option('display.max_columns', None)
pd.set_option('display.float_format', '{:.2f}'.format)

## Sección 0: Setup y Carga

In [66]:
DATA_PATH = Path('Challenge_DataAI_Candidatos.csv')

df = pd.read_csv(
    DATA_PATH,
    sep=';',
    encoding='latin-1',
    dtype={'CODIGO_ENTIDAD': str}
)

# Strip de espacios en nombres de columna
df.columns = df.columns.str.strip()

# Verificación inmediata de carga
print(f"Shape: {df.shape}")
print(f"\nTipos de dato:\n{df.dtypes}")
print(f"\nPrimeras filas:")
df.head(3)

/tmp/ipykernel_24584/3100169084.py:3: DtypeWarning: Columns (0: TIPO_ENTIDAD) have mixed types. Specify dtype option on import or set low_memory=False.
  df = pd.read_csv(


Shape: (1044109, 18)

Tipos de dato:
TIPO_ENTIDAD                            object
CODIGO_ENTIDAD                             str
NOMBRE_ENTIDAD                             str
FECHA_CORTE                                str
UNIDAD_CAPTURA                         float64
NOMBRE_UNIDAD_CAPTURA                      str
CODIGO_PRODUCTO                        float64
PRODUCTO                                   str
CODIGO_MOTIVO                          float64
MOTIVO                                     str
QUEJAS_PENDIENTES                      float64
QUEJAS_RECIBIDAS                       float64
QUEJAS_FINALIZADAS                     float64
QUEJAS_FINALIZADAS_N1                  float64
QUEJAS_FINALIZADAS_N2                  float64
QUEJAS_EN_TRAMITE                      float64
QUEJAS_FAVOR_CONSUM_ACEP_ENT           float64
QUEJAS_FAVOR_CONSUM_NOACEP_ENT,,,,,        str
dtype: object

Primeras filas:


,TIPO_ENTIDAD,CODIGO_ENTIDAD,NOMBRE_ENTIDAD,FECHA_CORTE,UNIDAD_CAPTURA,NOMBRE_UNIDAD_CAPTURA,CODIGO_PRODUCTO,PRODUCTO,CODIGO_MOTIVO,MOTIVO,QUEJAS_PENDIENTES,QUEJAS_RECIBIDAS,QUEJAS_FINALIZADAS,QUEJAS_FINALIZADAS_N1,QUEJAS_FINALIZADAS_N2,QUEJAS_EN_TRAMITE,QUEJAS_FAVOR_CONSUM_ACEP_ENT,"QUEJAS_FAVOR_CONSUM_NOACEP_ENT,,,,,"
0,14,11,Banco SevoraMERICANA S.A.,30/06/2023,2.00,DEFENSORES DEL CONSUMIDOR FINANCIERO,28.00,SEGURO DE SALUD,135.00,INCUMPLIMIENTO DE OBLIGACIONES EN PRESTACI??N ...,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"0,,,,,"
1,14,11,Banco SevoraMERICANA S.A.,31/03/2024,1.00,ENTIDAD VIGILADA,36.00,SEGURO DE RENTAS VOLUNTARIAS,20.00,"DEMORA O NO DEVOLUCI??N DE SALDOS, APORTES O P...",0.00,0.00,0.00,0.00,0.00,0.00,0.00,"0,,,,"
2,13,18,Banco OximiaMERICANA S.A.,30/09/2024,2.00,DEFENSORES DEL CONSUMIDOR FINANCIERO,7.00,SEGURO DE SUSTRACCI??N,13.00,DEMORA O NO ENTREGA DEL CONTRATO O DE LA P??LIZA,0.00,0.00,0.00,0.00,0.00,0.00,0.00,"0,,,,,"


In [67]:
df.columns = df.columns.str.strip().str.rstrip(',') #QUEJAS_FAVOR_CONSUM_NOACEP_ENT sin ','

In [68]:
df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT'] = (
    df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT']
    .str.rstrip(',')
    .str.strip()
    .replace('', np.nan)
    .astype(float)
)

In [69]:
df.isnull().sum(axis=1).value_counts().sort_index(ascending=False).head(3) #Filas nulas

17         16
0     1044093
Name: count, dtype: int64

### Constantes del proyecto:

In [70]:
# Columnas agrupadas por rol analítico
COLS_ID        = ['TIPO_ENTIDAD', 'CODIGO_ENTIDAD', 'NOMBRE_ENTIDAD']
COLS_TIEMPO    = ['FECHA_CORTE']
COLS_PRODUCTO  = ['CODIGO_PRODUCTO', 'PRODUCTO']
COLS_MOTIVO    = ['CODIGO_MOTIVO', 'MOTIVO']
COLS_CANAL     = ['UNIDAD_CAPTURA', 'NOMBRE_UNIDAD_CAPTURA']
COLS_QUEJAS = [
    'QUEJAS_PENDIENTES', 'QUEJAS_RECIBIDAS', 'QUEJAS_FINALIZADAS',
    'QUEJAS_FINALIZADAS_N1', 'QUEJAS_FINALIZADAS_N2',
    'QUEJAS_EN_TRAMITE', 'QUEJAS_FAVOR_CONSUM_ACEP_ENT',
    'QUEJAS_FAVOR_CONSUM_NOACEP_ENT'
]

# Thresholds documentados en el enunciado
OUTLIER_EN_TRAMITE = 5_000

# Ventana de análisis de riesgo
ANIO_INICIO_RIESGO = 2021

In [71]:
# ¿N1 + N2 == FINALIZADAS?
check1 = ((df['QUEJAS_FINALIZADAS_N1'] + df['QUEJAS_FINALIZADAS_N2']).round(0) == df['QUEJAS_FINALIZADAS'].round(0)).mean()

# ¿ACEP + NOACEP == alguna de las columnas de finalizadas?
check2 = ((df['QUEJAS_FAVOR_CONSUM_ACEP_ENT'] + df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT']).round(0) == df['QUEJAS_FINALIZADAS'].round(0)).mean()
check3 = ((df['QUEJAS_FAVOR_CONSUM_ACEP_ENT'] + df['QUEJAS_FAVOR_CONSUM_NOACEP_ENT']).round(0) == df['QUEJAS_FINALIZADAS_N2'].round(0)).mean()

print(f"N1 + N2 == FINALIZADAS: {check1:.2%}")
print(f"ACEP + NOACEP == FINALIZADAS: {check2:.2%}")
print(f"ACEP + NOACEP == N2: {check3:.2%}")

N1 + N2 == FINALIZADAS: 98.12%
ACEP + NOACEP == FINALIZADAS: 75.60%
ACEP + NOACEP == N2: 78.18%


## Hallazgos de Carga: Sección 0

| Hallazgo | Detalle | Acción |
|---|---|---|
| Trailing commas en nombre de columna | `QUEJAS_FAVOR_CONSUM_NOACEP_ENT,,,,,` | `str.rstrip(',')` en carga |
| Trailing commas en valores | `QUEJAS_FAVOR_CONSUM_NOACEP_ENT` tenía valores `'0,,,,,'`  cargada como string | Limpieza y conversión a float en carga |
| 16 filas completamente nulas | Nulos uniformes en todas las columnas | Eliminación en S1 |
| N1 y N2 son nivel de atención | N1 = primera línea, N2 = segunda línea (escaladas). N1+N2 = FINALIZADAS en 98.12% | Columnas se conservan con nombre original |
| ACEP + NOACEP no reconstruye FINALIZADAS | Coincidencia solo del 75.60% estas columnas representan un subconjunto, no el total | Investigar en S2 (EDA) |
| Columnas de resolución favor consumidor ausentes | El enunciado describe `QUEJAS_FINALIZAD_FAVOR_CONSUM` y `FAVOR_ENTIDAD`  no existen en el CSV | Variable objetivo se construirá desde `ACEP` y `NOACEP` decisión a justificar en S3 |

## Sección 1: Limpieza minima estructural

In [72]:
df_raw_shape = df.shape

Las 17 filas identificadas en S0 tienen 17/18 columnas nulas: el parser interpretó toda la fila como un string en `TIPO_ENTIDAD` por comillas dobles en el nombre de entidad. Se eliminan antes de cualquier análisis.

In [73]:
df.isnull().sum(axis=1).value_counts().sort_index(ascending=False)

17         16
0     1044093
Name: count, dtype: int64

In [74]:
mask_mal_parseadas = df.isnull().sum(axis=1) == 17
df = df[~mask_mal_parseadas].copy()

print(f"Filas eliminadas: {mask_mal_parseadas.sum()} | Shape final: {df.shape}")

Filas eliminadas: 16 | Shape final: (1044093, 18)


In [75]:
df['FECHA_CORTE'] = pd.to_datetime(df['FECHA_CORTE'], dayfirst=True, errors='coerce')
df['ANIO'] = df['FECHA_CORTE'].dt.year
df['TRIMESTRE'] = df['FECHA_CORTE'].dt.to_period('Q')

print(f"Fechas no parseadas: {df['FECHA_CORTE'].isna().sum()}")
print(df['ANIO'].value_counts().sort_index())

Fechas no parseadas: 0
ANIO
2014      9868
2015     14298
2016     15621
2017     16608
2018     16366
2019      9987
2020     16067
2021     16086
2022     14437
2023    225700
2024    303466
2025    308128
2026     77461
Name: count, dtype: int64


Normalización de strings:

In [76]:
df['NOMBRE_ENTIDAD'] = df['NOMBRE_ENTIDAD'].str.strip().str.upper()
df['PRODUCTO']       = df['PRODUCTO'].str.strip()
df['MOTIVO']         = df['MOTIVO'].str.strip()

___

In [77]:
# ¿Los duplicados incluyen FECHA_CORTE o son idénticos en todo incluyendo fecha?
cols_key = ['CODIGO_ENTIDAD', 'FECHA_CORTE', 'CODIGO_PRODUCTO', 'CODIGO_MOTIVO', 'UNIDAD_CAPTURA']
n_dupes_key = df.duplicated(subset=cols_key).sum()
print(f"Duplicados en clave natural: {n_dupes_key}")
print(f"Duplicados exactos (todas columnas): {df.duplicated().sum()}")

Duplicados en clave natural: 715305
Duplicados exactos (todas columnas): 609514


In [78]:
# Ver un ejemplo concreto de duplicado exacto
clave = ['CODIGO_ENTIDAD', 'FECHA_CORTE', 'CODIGO_PRODUCTO', 'CODIGO_MOTIVO', 'UNIDAD_CAPTURA']
ejemplo = df[df.duplicated(subset=clave, keep=False)].sort_values(clave).head(10)
print(ejemplo[clave + ['QUEJAS_RECIBIDAS', 'QUEJAS_FINALIZADAS']].to_string())

        CODIGO_ENTIDAD FECHA_CORTE  CODIGO_PRODUCTO  CODIGO_MOTIVO  UNIDAD_CAPTURA  QUEJAS_RECIBIDAS  QUEJAS_FINALIZADAS
431554           00001  2023-09-30             1.00         122.00            1.00              1.00                1.00
490907           00001  2023-09-30             1.00         122.00            1.00              2.00                2.00
364990           00001  2024-06-30             1.00         134.00            1.00              1.00                1.00
370842           00001  2024-06-30             1.00         134.00            1.00              1.00                1.00
1001930          00001  2024-06-30             1.00         134.00            1.00             13.00               12.00
129958           00001  2024-12-31             1.00          20.00            1.00              3.00                3.00
184646           00001  2024-12-31             1.00          20.00            1.00              4.00                5.00
684095           00001  2024-12-

In [79]:
# ¿Cuántas combinaciones clave tienen más de un registro?
clave = ['CODIGO_ENTIDAD', 'FECHA_CORTE', 'CODIGO_PRODUCTO', 'CODIGO_MOTIVO', 'UNIDAD_CAPTURA']
conteo = df.groupby(clave).size()
print(conteo.value_counts().sort_index())

1      276789
2       16311
3        2588
4        1214
5         778
        ...  
192         1
196         1
220         1
222         1
499         1
Name: count, Length: 145, dtype: int64


In [80]:
# ¿Los múltiples registros por clave vienen de distintas UNIDAD_CAPTURA o de la misma?
clave_sin_canal = ['CODIGO_ENTIDAD', 'FECHA_CORTE', 'CODIGO_PRODUCTO', 'CODIGO_MOTIVO']
df_multi = df.groupby(clave_sin_canal).filter(lambda x: len(x) > 1)
print(df_multi.groupby(clave_sin_canal)['UNIDAD_CAPTURA'].nunique().value_counts())

UNIDAD_CAPTURA
2    73117
1    17290
Name: count, dtype: int64


In [81]:
# Los 609.514 duplicados exactos sí se eliminan — filas idénticas en todas las columnas.
# Las combinaciones con múltiples registros y valores distintos NO se tocan:
# 73.117 corresponden a doble reporte por fuente (entidad vs defensor) — diseño del dataset.
# 17.290 son doble reporte de la misma fuente — se documentan como limitación en S2.

n_before = len(df)
df = df.drop_duplicates()
print(f"Duplicados exactos eliminados: {n_before - len(df)} | Shape final: {df.shape}")

Duplicados exactos eliminados: 609514 | Shape final: (434579, 20)


In [82]:
print(f"Shape final S1: {df.shape}")
print(f"\nNulidad residual:")
print(df.isnull().sum()[df.isnull().sum() > 0])
print(f"\nDtipos:")
print(df.dtypes)

Shape final S1: (434579, 20)

Nulidad residual:
Series([], dtype: int64)

Dtipos:
TIPO_ENTIDAD                              object
CODIGO_ENTIDAD                               str
NOMBRE_ENTIDAD                               str
FECHA_CORTE                       datetime64[us]
UNIDAD_CAPTURA                           float64
NOMBRE_UNIDAD_CAPTURA                        str
CODIGO_PRODUCTO                          float64
PRODUCTO                                     str
CODIGO_MOTIVO                            float64
MOTIVO                                       str
QUEJAS_PENDIENTES                        float64
QUEJAS_RECIBIDAS                         float64
QUEJAS_FINALIZADAS                       float64
QUEJAS_FINALIZADAS_N1                    float64
QUEJAS_FINALIZADAS_N2                    float64
QUEJAS_EN_TRAMITE                        float64
QUEJAS_FAVOR_CONSUM_ACEP_ENT             float64
QUEJAS_FAVOR_CONSUM_NOACEP_ENT           float64
ANIO                                

In [83]:
print(f"Filas:    {df_raw_shape[0]:>10,}  →  {df.shape[0]:>10,}  ({df_raw_shape[0] - df.shape[0]:,} eliminadas)")
print(f"Columnas: {df_raw_shape[1]:>10}  →  {df.shape[1]:>10}  (+ANIO, +TRIMESTRE)")

Filas:     1,044,109  →     434,579  (609,530 eliminadas)
Columnas:         18  →          20  (+ANIO, +TRIMESTRE)


Se eliminaron filas mal parseadas, duplicados exactos y se unificaron fechas y strings. El dataset pasa de 1.044.110 a 434.579 fila.

## S2: EDA